# Speech Understanding 
# Lecture 14: Text-to-Speech Synthesis

### Mark Hasegawa-Johnson, KCGI

1. <a href="#section1">Installing gTTs, the Google speech synthesizer</a>
1. <a href="#section2">Using gTTs</a>
1. <a href="#section3">Use SpeechRecognizer to check the output</a>
1. <a href="#homework">Homework</a>


<a id='section1'></a>

## 1. Installing gTTs, the Google speech synthesizer

For speech synthesis, we will use Google's text-to-speech synthesis system (gTTs).  You need to be connected to the internet in order to use it. Documentation for gTTs is here: https://gtts.readthedocs.io/en/latest/ 

gTTs is installed like this (either in the window below, or in a terminal):

In [1]:
!pip install gTTs

  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gTTs]


<a id="section2"></a>

## 2. Using gTTs

gTTs can't play the audio directly.  We need to create the audio output, save it to a file, and then play back the file.

In [7]:
import gtts, librosa, IPython

desired_text = "これは合成音声です"
#desired_text = "this is speech synthesis"
tts = gtts.gTTS(text=desired_text, lang="ja")
tts.save("speech.mp3")
    
speech_wave, speech_rate = librosa.load("speech.mp3")
IPython.display.Audio(data=speech_wave, rate=speech_rate)

<a id="section3"></a>

## 3. Use SpeechRecognizer to check the output

Often, in the real world, you need to generate synthetic speech prompts for a customer, but you don't have time to listen to all of them to make sure they're OK.

When that happens, you can use `SpeechRecognizer` to check each of the files automatically.  If `SpeechRecognizer` detects any mistake, you can check manually to see if it's OK.

First, we need to convert the file from `mp3` format to a format that `SpeechRecognizer` can handle.  `SpeechRecognizer` can handle wav and flac files.  We can use librosa to read in the mp3, but then to write it out, we need to install and use `soundfile`:

In [12]:
!pip install soundfile
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 6.1 MB/s  0:00:05 eta 0:00:010:00:01


In [13]:
import soundfile 
data, samplerate = librosa.load('speech.mp3')
soundfile.write('speech.wav',data,samplerate)

Now we can call `speech_recognition` to check the file content:

In [14]:
import speech_recognition

r = speech_recognition.Recognizer()

with speech_recognition.AudioFile("speech.wav") as source:
    audio = r.record(source)
    text = r.recognize_google(audio, language="ja")
    print('The person in this audio file said "%s"'%(text))
    if text == desired_text:
        print('This matches the desired text')
    else:
        print('This does not match the desired text, which was "%s"'%(desired_text))

The person in this audio file said "これは 合成音声です"
This does not match the desired text, which was "これは合成音声です"


It looks like it matched except for the space.  To prevent the space from causing a mismatch, we can remove all spaces from both strings:

In [15]:
r = speech_recognition.Recognizer()

with speech_recognition.AudioFile("speech.wav") as source:
    audio = r.record(source)
    text = r.recognize_google(audio, language="ja")
    print('The person in this audio file said "%s"'%(text))
    if text.replace(" ","") == desired_text.replace(" ",""):
        print('This matches the desired text')
    else:
        print('This does not match the desired text, which was "%s"'%(desired_text))

The person in this audio file said "これは 合成音声です"
This matches the desired text


<a id='homework'></a>

## Homework

Edit the text file called `homework14.py`.

This file should `def` a function called `synthesize`, with a signature as shown here:

In [16]:
import homework14, importlib
importlib.reload(homework14)
help(homework14.synthesize)

Help on function synthesize in module homework14:

synthesize(text, lang, filename)
    Use gtts.gTTS(text=text, lang=lang) to synthesize speech, then write it to filename.

    @params:
    text (str) - the text you want to synthesize
    lang (str) - the language in which you want to synthesize it
    filename (str) - the filename in which it should be saved



Test whether your code works by running the following block:

In [17]:
import homework14, librosa, IPython, importlib
importlib.reload(homework14)

homework14.synthesize("This is speech synthesis!","en","english.mp3")
y, sr = librosa.load("english.mp3")
IPython.display.Audio(data=y, rate=sr)

Now let's try using the speech synthesizer to create a lot of speech files:

In [18]:
importlib.reload(homework14)
help(homework14.make_a_corpus)

Help on function make_a_corpus in module homework14:

make_a_corpus(texts, languages, filenames)
    Create many speech files, and check their content using SpeechRecognition.
    The output files should be created as MP3, then converted to WAV, then recognized.

    @param:
    texts - a list of the texts you want to synthesize
    languages - a list of their languages
    filenames - a list of their root filenames, without the ".mp3" ending

    @return:
    recognized_texts - list of the strings that were recognized from each file



In [21]:
importlib.reload(homework14)

texts = [
    'hello',
    'my name is',
    'my name is',
    'will the real slim shady please stand up'
]
languages = ['en','en','en','en']
filenames = ['file1','file2','file3','file4']
recognized_texts = homework14.make_a_corpus(texts, languages, filenames)
print(texts)
print(recognized_texts)
for text, recognized_text in zip(texts, recognized_texts):
    print(text,'was recognized as',recognized_text)

['hello', 'my name is', 'my name is', 'will the real slim shady please stand up']
['hello', 'my name is', 'my name is', 'will The Real Slim Shady please stand up']
hello was recognized as hello
my name is was recognized as my name is
my name is was recognized as my name is
will the real slim shady please stand up was recognized as will The Real Slim Shady please stand up


### Receiving your grade

In order to receive a grade for your homework, you need to:

1. Run the following code block on your machine.  The result may list some errors, and then in the very last line, it will show a score.  That score (between 0% and 100%) is the grade you have earned so far.  If you want to earn a higher grade, please continue editing `homework3.py`, and then run this code block again.
1. When you are happy with your score (e.g., when it reaches 100%), choose `File` $\Rightarrow$ `Save and Checkpoint`.  Then use `GitHub Desktop` to commit and push your changes.
1. Make sure that the 100% shows on your github repo on github.com.  If it doesn't, you will not receive credit.

In [22]:
import importlib, grade
importlib.reload(grade)

/Users/jhasegaw/kcgi/intro_speech_understanding/lec14/grade.py:13: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(os.path.isfile("english.mp3"), "english.mp3 was not created")
/Users/jhasegaw/kcgi/intro_speech_understanding/lec14/grade.py:17: SyntaxWarning: assertion is always true, perhaps remove parentheses?
  assert(isinstance(recog,list),'make_a_corpus should return a list')
...
----------------------------------------------------------------------
Ran 3 tests in 1.312s

OK


3 successes out of 3 tests run
Score: 100%


...
----------------------------------------------------------------------
Ran 3 tests in 0.990s

OK


3 successes out of 3 tests run
Score: 100%


<module 'grade' from '/Users/jhasegaw/kcgi/intro_speech_understanding/lec14/grade.py'>